In [1]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)

print(f"La root del progetto è: {PROJECT_ROOT}")

La root del progetto è: /home/cvalentino/SissaUnisaDraftCodes/


In [2]:
from paths import INV_PATH, TP2_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"

Import delle librerie

In [3]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

Fissiamo precisione doppia

In [4]:
torch.set_default_dtype(torch.float64)

## Creazione dei dati per il problema inverso

Fissiamo i punti in cui sono installati i sensori

In [5]:
column = Blend2Pina(LOAD_MODEL + model_name)

Read blend: "/home/cvalentino/SissaUnisaDraftCodes/models/column/column.blend"


Acquisizione dei punti al contrno

In [6]:
num_points = 1_000

surface = column.boundary(time_interval=[0, 1])
points = surface.sample(num_points)

Fissimao i parametri e collezioniamo i dati simulati

In [7]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u1 = torch.exp(par_lambda*points.extract('t')) + par_alpha*points.extract('x') + par_beta*points.extract('y') + points.extract('z')
u2 = torch.exp(par_lambda*points.extract('t')) + par_alpha*(points.extract('x')**2) + par_beta*(points.extract('y')**2) + points.extract('z')**2

Creazione file .csv per conservare i punti al contorno

In [8]:
total_info = torch.concat(
    [points.tensor, u1, u2],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "t", "u1", "u2"]
)

df.to_csv("./files/data.csv", sep=";")

## Creazione mesh e xdmf

In [9]:
column_msh = Blend2Mesh(LOAD_MODEL + model_name, "column")

Read blend: "/home/cvalentino/SissaUnisaDraftCodes/models/column/column.blend"


In [10]:
column_msh.create_single_meshes(len_msh=0.01)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 10%] Meshing curve 4 (Line)
Info    : [ 20%] Meshing curve 5 (Line)
Info    : [ 20%] Meshing curve 6 (Line)
Info    : [ 20%] Meshing curve 7 (Line)
Info    : [ 20%] Meshing curve 8 (Line)
Info    : [ 30%] Meshing curve 9 (Line)
Info    : [ 30%] Meshing curve 10 (Line)
Info    : [ 30%] Meshing curve 11 (Line)
Info    : [ 40%] Meshing curve 12 (Line)
Info    : [ 40%] Meshing curve 13 (Line)
Info    : [ 40%] Meshing curve 14 (Line)
Info    : [ 40%] Meshing curve 15 (Line)
Info    : [ 50%] Meshing curve 16 (Line)
Info    : [ 50%] Meshing curve 17 (Line)
Info    : [ 50%] Meshing curve 18 (Line)
Info    : [ 50%] Meshing curve 19 (Line)
Info    : [ 60%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 60%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 70%] Meshing curve 24 (Line)
I

In [11]:
column_xdmf = Msh2Xdmf("column.msh", "column")
column_xdmf.to_xdmf(num_refine=3)
column_xdmf.to_xdmf()

Info    : Reading 'column.msh'...
Info    : 345 entities
Info    : 62 nodes
Info    : 125 elements
Info    : Done reading 'column.msh'
Info    : Meshing 1D...
Info    : Done meshing 1D (Wall 4.5882e-05s, CPU 5e-05s)
Info    : Meshing 2D...
Info    : Done meshing 2D (Wall 3.2291e-05s, CPU 3.5e-05s)
Info    : Meshing 3D...
Info    : Done meshing 3D (Wall 0.000123327s, CPU 0.000124s)
Info    : Optimizing mesh...
Info    : Done optimizing mesh (Wall 1.46109e-05s, CPU 2.1e-05s)
Info    : 62 nodes 165 elements
Info    : Refining mesh...
Info    : Meshing order 2 (curvilinear on)...
Info    : [  0%] Meshing curve 1 order 2
Info    : [ 10%] Meshing curve 2 order 2
Info    : [ 10%] Meshing curve 3 order 2
Info    : [ 10%] Meshing curve 4 order 2
Info    : [ 10%] Meshing curve 5 order 2
Info    : [ 10%] Meshing curve 6 order 2
Info    : [ 10%] Meshing curve 7 order 2
Info    : [ 10%] Meshing curve 8 order 2
Info    : [ 10%] Meshing curve 9 order 2
Info    : [ 10%] Meshing curve 10 order 2
Info  